# Game Analytics: Database Setup & Population
## Notebook 2 - PostgreSQL Schema Creation & Data Insertion

This notebook handles:
1. **Database Connection** - Connect to PostgreSQL via SQLAlchemy
2. **Schema Creation** - Create all 6 tables with proper PK/FK constraints
3. **Data Insertion** - Populate tables from the extracted DataFrames
4. **Verification** - Run test queries to confirm data is loaded

### Tables Created:
| Table | Primary Key | Foreign Keys |
|-------|-------------|--------------|
| categories | category_id | -- |
| competitions | competition_id | category_id, parent_id (self) |
| complexes | complex_id | -- |
| venues | venue_id | complex_id |
| competitors | competitor_id | -- |
| competitor_rankings | rank_id (SERIAL) | competitor_id |

Database: PostgreSQL | ORM: SQLAlchemy

## 1. Import Libraries & Configuration

In [ ]:
import os
import pandas as pd
from sqlalchemy import create_engine, text
from sqlalchemy.exc import SQLAlchemyError

# Database Configuration (PostgreSQL)
DB_USER = os.getenv("DB_USER", "postgres")
DB_PASSWORD = os.getenv("DB_PASSWORD", "your_password")
DB_HOST = os.getenv("DB_HOST", "localhost")
DB_PORT = os.getenv("DB_PORT", "5432")
DB_NAME = os.getenv("DB_NAME", "tennis_db")

DATABASE_URL = f"postgresql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"

print(f"Database URL: postgresql://{DB_USER}:***@{DB_HOST}:{DB_PORT}/{DB_NAME}")

## 2. Create Database Engine

First, we connect to the PostgreSQL server and create the `tennis_db` database if it doesn't exist.

In [ ]:
server_url = f"postgresql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/postgres"
server_engine = create_engine(server_url, isolation_level="AUTOCOMMIT")

try:
    with server_engine.connect() as conn:
        result = conn.execute(
            text("SELECT 1 FROM pg_database WHERE datname = :dbname"),
            {"dbname": DB_NAME}
        )
        if result.fetchone():
            print(f"Database '{DB_NAME}' already exists.")
        else:
            conn.execute(text(f'CREATE DATABASE {DB_NAME}'))
            print(f"Database '{DB_NAME}' created successfully.")
except SQLAlchemyError as e:
    print(f"Error creating database: {e}")
finally:
    server_engine.dispose()

In [ ]:
engine = create_engine(DATABASE_URL, pool_pre_ping=True)

with engine.connect() as conn:
    version = conn.execute(text("SELECT version()")).scalar()
    print(f"Connected to PostgreSQL:")
    print(f"  {version}")

## 3. Create Schema (All 6 Tables)

Execute the DDL statements to create tables with proper primary keys, foreign keys, and indexes.

In [ ]:
drop_statements = [
    "DROP TABLE IF EXISTS competitor_rankings CASCADE;",
    "DROP TABLE IF EXISTS competitors CASCADE;",
    "DROP TABLE IF EXISTS venues CASCADE;",
    "DROP TABLE IF EXISTS complexes CASCADE;",
    "DROP TABLE IF EXISTS competitions CASCADE;",
    "DROP TABLE IF EXISTS categories CASCADE;",
]

with engine.connect() as conn:
    for stmt in drop_statements:
        conn.execute(text(stmt))
    conn.commit()
    print("Dropped existing tables (if any).")

In [ ]:
create_statements = [
    """CREATE TABLE categories (
        category_id    VARCHAR(50)  PRIMARY KEY,
        category_name  VARCHAR(100) NOT NULL
    );""",
    """CREATE TABLE competitions (
        competition_id   VARCHAR(50)  PRIMARY KEY,
        competition_name VARCHAR(100) NOT NULL,
        parent_id        VARCHAR(50)  NULL,
        type             VARCHAR(20)  NOT NULL,
        gender           VARCHAR(10)  NOT NULL,
        category_id      VARCHAR(50)  NOT NULL,
        FOREIGN KEY (category_id) REFERENCES categories(category_id),
        FOREIGN KEY (parent_id) REFERENCES competitions(competition_id)
    );""",
    """CREATE TABLE complexes (
        complex_id    VARCHAR(50)  PRIMARY KEY,
        complex_name  VARCHAR(100) NOT NULL
    );""",
    """CREATE TABLE venues (
        venue_id      VARCHAR(50)  PRIMARY KEY,
        venue_name    VARCHAR(100) NOT NULL,
        city_name     VARCHAR(100) NOT NULL,
        country_name  VARCHAR(100) NOT NULL,
        country_code  CHAR(3)      NOT NULL,
        timezone      VARCHAR(100) NOT NULL,
        complex_id    VARCHAR(50)  NOT NULL,
        FOREIGN KEY (complex_id) REFERENCES complexes(complex_id)
    );""",
    """CREATE TABLE competitors (
        competitor_id  VARCHAR(50)  PRIMARY KEY,
        name           VARCHAR(100) NOT NULL,
        country        VARCHAR(100) NOT NULL,
        country_code   CHAR(3)      NOT NULL,
        abbreviation   VARCHAR(10)  NOT NULL
    );""",
    """CREATE TABLE competitor_rankings (
        rank_id             SERIAL PRIMARY KEY,
        rank                INT          NOT NULL,
        movement            INT          NOT NULL,
        points              INT          NOT NULL,
        competitions_played INT          NOT NULL,
        competitor_id       VARCHAR(50)  NOT NULL,
        FOREIGN KEY (competitor_id) REFERENCES competitors(competitor_id)
    );""",
    "CREATE INDEX idx_competitions_category  ON competitions(category_id);",
    "CREATE INDEX idx_competitions_type       ON competitions(type);",
    "CREATE INDEX idx_venues_complex          ON venues(complex_id);",
    "CREATE INDEX idx_venues_country          ON venues(country_name);",
    "CREATE INDEX idx_rankings_competitor     ON competitor_rankings(competitor_id);",
    "CREATE INDEX idx_rankings_rank           ON competitor_rankings(rank);",
    "CREATE INDEX idx_competitors_country     ON competitors(country);",
]

with engine.connect() as conn:
    for stmt in create_statements:
        clean = stmt.strip()
        if clean:
            conn.execute(text(clean))
    conn.commit()
    print("All 6 tables created with PK/FK constraints and indexes.")

## 4. Load Extracted Data

Load the DataFrames from CSV files (saved by Notebook 1) or re-extract from the API.

In [ ]:
import os

data = {}
csv_files = {
    "categories": "data/categories.csv",
    "competitions": "data/competitions.csv",
    "complexes": "data/complexes.csv",
    "venues": "data/venues.csv",
    "competitors": "data/competitors.csv",
    "competitor_rankings": "data/competitor_rankings.csv",
}

for name, path in csv_files.items():
    if os.path.exists(path):
        data[name] = pd.read_csv(path)
        print(f"Loaded {name}: {len(data[name])} rows from {path}")
    else:
        print(f"{path} not found. Run Notebook 1 first or re-extract from API.")
        data[name] = pd.DataFrame()

# Option B: If CSVs dont exist, re-extract from API
# from data_extraction import TennisDataExtractor
# extractor = TennisDataExtractor()
# data = extractor.extract_all()

## 5. Handle Missing Values

Some fields like `country_code` may be missing for certain competitors. We fill them with placeholder values to satisfy NOT NULL constraints.

In [ ]:
# Handle missing values for SQL NOT NULL constraints
if not data["competitors"].empty:
    data["competitors"]["country_code"] = data["competitors"]["country_code"].fillna("XXX")
    data["competitors"]["abbreviation"] = data["competitors"]["abbreviation"].fillna("N/A")
    data["competitors"]["country"] = data["competitors"]["country"].fillna("Unknown")
    data["competitors"]["name"] = data["competitors"]["name"].fillna("Unknown")

if not data["competitions"].empty:
    data["competitions"]["type"] = data["competitions"]["type"].fillna("unknown")
    data["competitions"]["gender"] = data["competitions"]["gender"].fillna("unknown")

print("Missing values handled.")

## 6. Insert Data into PostgreSQL

Insertion order respects foreign-key dependencies:
1. categories (no FK)
2. competitions (FK -> categories)
3. complexes (no FK)
4. venues (FK -> complexes)
5. competitors (no FK)
6. competitor_rankings (FK -> competitors)

In [ ]:
table_order = [
    "categories", "competitions", "complexes",
    "venues", "competitors", "competitor_rankings",
]

table_name_map = {
    "categories": "categories",
    "competitions": "competitions",
    "complexes": "complexes",
    "venues": "venues",
    "competitors": "competitors",
    "competitor_rankings": "competitor_rankings",
}

for key in table_order:
    df = data.get(key)
    if df is None or df.empty:
        print(f"  Skipping '{key}' -- no data available.")
        continue
    table_name = table_name_map[key]
    try:
        df.to_sql(name=table_name, con=engine, if_exists="append",
                  index=False, method="multi", chunksize=500)
        print(f"  Inserted {len(df)} rows into {table_name}")
    except SQLAlchemyError as e:
        print(f"  Failed to insert into {table_name}: {e}")

## 7. Verify Data - Run Test Queries

In [ ]:
verification_queries = {
    "categories": "SELECT COUNT(*) FROM categories;",
    "competitions": "SELECT COUNT(*) FROM competitions;",
    "complexes": "SELECT COUNT(*) FROM complexes;",
    "venues": "SELECT COUNT(*) FROM venues;",
    "competitors": "SELECT COUNT(*) FROM competitors;",
    "competitor_rankings": "SELECT COUNT(*) FROM competitor_rankings;",
}

print("=== Table Row Counts ===\n")
with engine.connect() as conn:
    for table, query in verification_queries.items():
        count = conn.execute(text(query)).scalar()
        print(f"  {table:25s} -> {count:>6} rows")

In [ ]:
sample_queries = {
    "categories (top 5)": "SELECT * FROM categories LIMIT 5;",
    "competitions (top 5)": "SELECT * FROM competitions LIMIT 5;",
    "complexes (top 5)": "SELECT * FROM complexes LIMIT 5;",
    "venues (top 5)": "SELECT * FROM venues LIMIT 5;",
    "competitors (top 5)": "SELECT * FROM competitors LIMIT 5;",
    "competitor_rankings (top 5)": "SELECT * FROM competitor_rankings LIMIT 5;",
}

for label, query in sample_queries.items():
    print(f"\n{'='*60}")
    print(f"  {label}")
    print(f"{'='*60}")
    df = pd.read_sql(text(query), engine)
    print(df.to_string(index=False))

## Database Setup Complete!

All 6 tables are created and populated with data from the Sportradar API.

### Next Steps
-> Open Notebook 3 - SQL Analysis Queries to run the 20 analysis queries.
-> Open Notebook 4 - Streamlit App to launch the interactive dashboard.

### Alternative: Use sample_data.sql
If you prefer to load data via SQL instead of Python:
```bash
psql -U postgres -d tennis_db -f schema.sql
psql -U postgres -d tennis_db -f sample_data.sql
```